# Chapter 2 &mdash; The Language Operations, Animated

**Concept 17 of the Chapter 2 decomposition:** *The Language Operations, Animated*

Two languages, two sliders, and colour that survives the operations &mdash; star shows its own decomposition, concatenation is blue then red, and a union says which side each string came from.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter2-Lang/Concept-Language-Operations-Animated/Concept-Language-Operations-Animated.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.AnimateLang    import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateLang as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateLang, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Chapter 2 defines five operations on languages and, until this notebook, offered
nothing to run. Sixteen concepts of set notation, in a course whose whole claim is
that you can **execute** the theory.

Here are the operations, moving.

`AnimateLang()` takes two languages. Slider $m$ builds $star(L_1,m)$, slider $n$ builds
$star(L_2,n)$, and **every** row is computed on those &mdash; so moving one slider
regrows union, concatenation, intersection and both stars at once. The play buttons
sweep a slider hands-free.

**Colour is the argument, not decoration.**

* $L_1$'s strings are **blue**, $L_2$'s are **red**.
* In a star, each string is drawn with its **pieces** in alternating shades, so you
  see *which elements were concatenated to build it* rather than having to work it
  out. The decomposition is the thing $star$ actually means.
* A concatenation is literally **blue pieces then red pieces** &mdash; which is what
  concatenation *is*.
* A union colours each string by which side supplied it, and **purple** when both
  did. The overlap becomes a colour instead of a claim.
* An intersection is therefore all purple, necessarily.

The sets grow like $|L|^m$. The box caps what it draws and always reports the true
size beside it; watching the count race past the box is the point, not a limitation.

Type your own operation into the last field &mdash; any `lambda A, B: ...` over the
two starred sets &mdash; and it is drawn with the same provenance colouring.

## 2. Definitions

### The two languages, and what star does to them

In [ ]:
L1 = {'a', 'b', 'bc', 'def'}       # blue
L2 = {'ab', 'c', 'cdef'}           # red

print('L1 =', sorted(L1, key=lambda s: (len(s), s)))
print('L2 =', sorted(L2, key=lambda s: (len(s), s)))
print()
print('%-4s %12s %12s' % ('m', '|star(L1,m)|', '|star(L2,m)|'))
for m in range(5):
    print('%-4d %12d %12d' % (m, len(lstar(L1, m)), len(lstar(L2, m))))
print()
print('Growth is roughly |L|^m.  That is why the display counts rather than')
print('promising to show you everything.')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch2&nbsp;16.&nbsp;Slippery Roads: Telling Look-Alike Language Definitions Apart](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter2-Lang/Concept-Slippery-Roads/Concept-Slippery-Roads.ipynb) &nbsp;&middot;&nbsp; [**Chapter 2** index](https://github.com/ganeshutah/Jove/blob/master/Chapter2-Lang/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;1.&nbsp;Star: Three Equivalent Definitions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3-Star/Concept-Star-Three-Definitions/Concept-Star-Three-Definitions.ipynb)&nbsp;&rarr;

---

## 3. Tests

**The decomposition is real.** `star_pieces` keeps, for each string, one way of building it out of elements &mdash; and those are the pieces the animation colours.

In [ ]:
from jove.AnimateLang import star_pieces

A = star_pieces(L1, 2)
for s in sorted(A, key=lambda s: (len(s), s))[:8]:
    print('   %-8s = %s' % (repr(s) if s else "''", ' + '.join(A[s]) or 'the empty concatenation'))

for s, pieces in A.items():
    assert ''.join(pieces) == s
    assert all(p in L1 for p in pieces)
    assert len(pieces) <= 2
print()
print('Checked: every decomposition concatenates back to its string, uses only')
print('elements of L1, and has at most m of them.')

And it agrees with Jove's own `lstar`, which is the definition it is animating.

In [ ]:
for m in range(5):
    assert set(star_pieces(L1, m)) == set(lstar(L1, m))
    assert set(star_pieces(L2, m)) == set(lstar(L2, m))
print('star_pieces and lstar agree on L1 and L2 for every m up to 4.')
print()
print('The animation adds the PIECES; it does not change the set.')

**Where the purple comes from.** A string is purple when both sides have it &mdash; and a string can land in both by being built two different ways.

In [ ]:
A, B = lstar(L1, 2), lstar(L2, 1)
both = sorted(A & B, key=lambda s: (len(s), s))
print('in both  :', [s or 'eps' for s in both])
print()
print("'ab' is in star(L1,2) as  a + b   (two blue pieces)")
print("     and in star(L2,1) as ab      (one red element)")
print()
print('Same string, two constructions.  The language does not care which; that')
print('is exactly the point Concept 16 makes about notation versus membership.')
assert 'ab' in both and '' in both

**Concatenation is not multiplication.** $|A\\cdot B|$ is at most $|A|\\times|B|$, and it is smaller whenever two different pairs collide on the same string.

In [ ]:
cat = lcat(A, B)
print('|A| = %d, |B| = %d, |A| x |B| = %d' % (len(A), len(B), len(A) * len(B)))
print('|A . B| = %d' % len(cat))
print()
collisions = len(A) * len(B) - len(cat)
print('%d pair(s) produced a string some other pair had already made.' % collisions)
print('In the animation those are the chips you do not see appear when you')
print('expected one -- the box grows more slowly than the arithmetic.')
assert len(cat) <= len(A) * len(B)

## 4. Animation

Drag either slider and watch every row regrow, or press play and let
it sweep. Edit `L1` and `L2` in the boxes at the top &mdash; a bare comma, or `eps`,
means the empty string. The last field takes any `lambda A, B: ...` over the two
starred sets.

In [ ]:
from jove.AnimateLang import *
AnimateLang()

## 5. Exercises


1. Set $m=0$. What is $star(L_1,0)$, and why is the concatenation row not empty?
2. Put the **same** language in both boxes. What happens to the union, and what
   happens to the intersection? Which one becomes the boring row?
3. Make $L_1=\{\varepsilon\}$ &mdash; type a bare `eps`. Predict all six rows
   before moving a slider, then check. This is Concept 11's `Unit()` seen from the
   side.
4. Make $L_2$ empty by clearing the box. Which rows go empty, and which do not?
   Concept 10 says why.
5. Find two different $L_1$ and $L_2$ whose **intersection** is bigger than either
   star alone would suggest &mdash; that is, arrange lots of purple.
6. In the last field, type `lambda A, B: {s for s in A if s[::-1] in A}`. What
   language is that, and what is it called in Chapter 3?
7. Set $L_1=\{0,1\}$ and sweep $m$. At which $m$ does the count first exceed what
   the box will draw, and what is $|star(\{0,1\},m)|$ in closed form?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter2-Lang/Concept-Language-Operations-Animated')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')